# Phase 1: Behavioral Feature-Performance Correlation
Determine which of the 32+ behavioral metrics correlate with and predict algorithm performance (AOCC), both globally and per-model.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyArrowPatch
from scipy import stats
from scipy.spatial import ConvexHull
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import silhouette_score
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("WARNING: shap not installed. SHAP cells will be skipped.")

plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 11})

# --- Constants ---
RESULTS_DIR = Path("../results_phase1")
MODELS = [
    "qwen3.5-4b", "qwen3.5-9b", "qwen3.5-27b",
    "rnj-1-8b", "devstral-small-2-24b",
    "olmo3-7b", "olmo3-32b", "granite4-3b",
    "gemini-3-pro", "gemini-3-flash",
]
N_SEEDS = 5
BUDGET = 100
N_INSTANCES = 10
EVAL_SEEDS = 5

FEATURE_CATEGORIES = {
    "Exploration & Diversity": [
        "avg_nearest_neighbor_distance", "dispersion", "avg_exploration_pct",
    ],
    "Exploitation": [
        "avg_distance_to_best", "intensification_ratio", "avg_exploitation_pct",
    ],
    "Convergence": [
        "average_convergence_rate", "avg_improvement", "success_rate",
    ],
    "Stagnation": [
        "longest_no_improvement_streak", "last_improvement_fraction",
    ],
    "Step-Size & Movement": [
        "step_size_mean", "step_size_std", "step_size_trend", "directional_persistence",
    ],
    "Information-Theoretic": [
        "fitness_sample_entropy", "fitness_permutation_entropy",
        "fitness_autocorrelation_lag1", "fitness_lempel_ziv_complexity",
        "fitness_hurst_exponent", "fitness_dfa_alpha",
    ],
    "Early/Late Dynamics": [
        "x_spread_early", "x_spread_late", "spread_ratio", "centroid_drift",
        "f_range_early", "f_range_late", "f_range_ratio",
    ],
    "Novel Features": [
        "improvement_spatial_correlation", "improvement_burstiness",
        "dimension_convergence_heterogeneity", "step_size_autocorrelation",
        "fitness_plateau_fraction", "half_convergence_time",
    ],
}

CATEGORY_COLORS = {
    "Exploration & Diversity": "#1f77b4",
    "Exploitation": "#ff7f0e",
    "Convergence": "#2ca02c",
    "Stagnation": "#d62728",
    "Step-Size & Movement": "#9467bd",
    "Information-Theoretic": "#8c564b",
    "Early/Late Dynamics": "#e377c2",
    "Novel Features": "#7f7f7f",
}


def parse_fitness(val):
    try:
        return float(val)
    except (TypeError, ValueError):
        return float("-inf")


def load_run(model_tag, seed):
    """Load log.jsonl for a given model and seed, return list of dicts."""
    seed_dir = RESULTS_DIR / model_tag / f"seed-{seed}"
    run_dirs = sorted(seed_dir.glob("run-*"))
    if not run_dirs:
        return []
    log_file = run_dirs[-1] / "log.jsonl"  # take latest run
    if not log_file.exists():
        return []
    entries = []
    with open(log_file, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                entries.append(json.loads(line))
    return entries


def load_all():
    """Load all results into a single DataFrame."""
    rows = []
    for model in MODELS:
        for seed in range(N_SEEDS):
            entries = load_run(model, seed)
            for i, entry in enumerate(entries):
                fitness = parse_fitness(entry.get("fitness"))
                meta = entry.get("metadata", {}) or {}
                aucs = meta.get("aucs", [])
                bf = meta.get("behavioral_features", {}) or {}
                failed = (fitness == float("-inf"))
                row = {
                    "model": model,
                    "seed": seed,
                    "evaluation": i,
                    "fitness": fitness,
                    "failed": failed,
                    "name": entry.get("name", ""),
                    "generation": entry.get("generation", None),
                    "auc_mean": np.mean(aucs) if aucs else np.nan,
                    "auc_std": np.std(aucs) if aucs else np.nan,
                }
                for k, v in bf.items():
                    row[f"bm_{k}"] = v
                rows.append(row)
    return pd.DataFrame(rows)


# --- Load data ---
df = load_all()
df_valid = df[~df["failed"]].copy()
# Keep only rows that have at least some behavioral data
bm_cols_all = [c for c in df_valid.columns if c.startswith("bm_")]
df_valid = df_valid.dropna(subset=bm_cols_all, how="all").reset_index(drop=True)

# Auto-discover bm_* columns
bm_cols = [c for c in df_valid.columns if c.startswith("bm_")]

# Build feature-to-category mapping
feat_to_cat = {}
for cat, feats in FEATURE_CATEGORIES.items():
    for f in feats:
        feat_to_cat[f"bm_{f}"] = cat

# Drop metrics that are NaN for >50% of candidates
nan_frac = df_valid[bm_cols].isna().mean()
keep_cols = nan_frac[nan_frac <= 0.5].index.tolist()
dropped = set(bm_cols) - set(keep_cols)
if dropped:
    print(f"Dropped {len(dropped)} metrics (>50% NaN): {sorted(dropped)}")
bm_cols = keep_cols

print(f"Valid candidates: {len(df_valid)}")
print(f"Behavioral metrics retained: {len(bm_cols)}")
print(f"Models found: {df_valid['model'].nunique()}")
print(f"\nCandidates per model:")
print(df_valid.groupby("model").size().to_string())

In [ ]:
# === Cell 2: Global Correlation Matrix ===

# Spearman correlation among all retained metrics + fitness
corr_cols = bm_cols + ["fitness"]
corr_data = df_valid[corr_cols].dropna()
spearman_corr = corr_data.corr(method="spearman")

# --- Full heatmap clustered by category ---
# Sort columns by category
cat_order = list(FEATURE_CATEGORIES.keys())
def sort_key(col):
    cat = feat_to_cat.get(col, "zzz")
    try:
        return (cat_order.index(cat), col)
    except ValueError:
        return (len(cat_order), col)

sorted_bm = sorted(bm_cols, key=sort_key)
sorted_all = sorted_bm + ["fitness"]

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(spearman_corr.loc[sorted_all, sorted_all].values,
               cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
labels = [c.replace("bm_", "") for c in sorted_all]
ax.set_xticks(range(len(sorted_all)))
ax.set_yticks(range(len(sorted_all)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8, label="Spearman rho")
ax.set_title("Full Spearman Correlation Matrix (clustered by category)")
plt.tight_layout()
plt.show()

# --- Bar chart: Spearman rho of each metric with AOCC ---
rho_with_aocc = []
for col in bm_cols:
    valid = corr_data[[col, "fitness"]].dropna()
    if len(valid) > 3:
        r, p = stats.spearmanr(valid[col], valid["fitness"])
        rho_with_aocc.append({"feature": col, "rho": r, "p": p,
                              "category": feat_to_cat.get(col, "Unknown")})

rho_df = pd.DataFrame(rho_with_aocc).sort_values("rho", key=abs, ascending=False)
n_tests = len(rho_df)
rho_df["p_bonf"] = rho_df["p"] * n_tests
rho_df["sig"] = rho_df["p_bonf"].apply(
    lambda p: "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
)

fig, ax = plt.subplots(figsize=(14, 8))
colors = [CATEGORY_COLORS.get(row["category"], "#333333") for _, row in rho_df.iterrows()]
bars = ax.barh(range(len(rho_df)), rho_df["rho"].values, color=colors)
labels_bar = [r.replace("bm_", "") for r in rho_df["feature"]]
ax.set_yticks(range(len(rho_df)))
ax.set_yticklabels(labels_bar, fontsize=9)
for i, (_, row) in enumerate(rho_df.iterrows()):
    offset = 0.01 if row["rho"] >= 0 else -0.01
    ha = "left" if row["rho"] >= 0 else "right"
    ax.text(row["rho"] + offset, i, row["sig"], va="center", ha=ha, fontsize=8)
ax.set_xlabel("Spearman rho with AOCC")
ax.set_title("Behavioral Feature Correlation with AOCC (Bonferroni-corrected)")
ax.axvline(0, color="k", lw=0.5)
# Legend for categories
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=cat) for cat, c in CATEGORY_COLORS.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# --- Inter-metric correlation heatmap (identify redundancy) ---
inter_corr = spearman_corr.loc[sorted_bm, sorted_bm]
fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(inter_corr.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
labels_inter = [c.replace("bm_", "") for c in sorted_bm]
ax.set_xticks(range(len(sorted_bm)))
ax.set_yticks(range(len(sorted_bm)))
ax.set_xticklabels(labels_inter, rotation=90, fontsize=8)
ax.set_yticklabels(labels_inter, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.8, label="Spearman rho")
ax.set_title("Inter-Metric Correlation (identify redundant features)")
plt.tight_layout()
plt.show()

# Print highly correlated pairs
print("\nHighly correlated pairs (|rho| > 0.8):")
for i, c1 in enumerate(sorted_bm):
    for j, c2 in enumerate(sorted_bm):
        if i < j and abs(inter_corr.loc[c1, c2]) > 0.8:
            print(f"  {c1.replace('bm_','')} <-> {c2.replace('bm_','')}: {inter_corr.loc[c1, c2]:.3f}")

In [ ]:
# === Cell 3: Per-Model Correlation ===

# Sort features by global |rho|
global_order = rho_df.sort_values("rho", key=abs, ascending=False)["feature"].tolist()

# Compute Spearman rho per (metric, model)
per_model_rho = pd.DataFrame(index=global_order, columns=MODELS, dtype=float)
for model in MODELS:
    sub = df_valid[df_valid["model"] == model]
    for feat in global_order:
        valid = sub[[feat, "fitness"]].dropna()
        if len(valid) > 3:
            r, _ = stats.spearmanr(valid[feat], valid["fitness"])
            per_model_rho.loc[feat, model] = r
        else:
            per_model_rho.loc[feat, model] = np.nan

fig, ax = plt.subplots(figsize=(14, 12))
data_plot = per_model_rho.astype(float).values
im = ax.imshow(data_plot, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(MODELS)))
ax.set_yticks(range(len(global_order)))
ax.set_xticklabels(MODELS, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels([f.replace("bm_", "") for f in global_order], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label="Spearman rho with AOCC")
ax.set_title("Per-Model Spearman Correlation with AOCC")
plt.tight_layout()
plt.show()

# Consistency score
consistency = per_model_rho.astype(float).std(axis=1).sort_values()
print("Consistency score (std of rho across models, lower = more consistent):")
for feat, std_val in consistency.items():
    label = feat.replace("bm_", "")
    mean_rho = per_model_rho.loc[feat].astype(float).mean()
    print(f"  {label:45s}  mean_rho={mean_rho:+.3f}  std={std_val:.3f}")

In [ ]:
# === Cell 4: Random Forest Feature Importance ===

# Prepare data
rf_data = df_valid[bm_cols + ["fitness"]].dropna()
X_rf = rf_data[bm_cols].values
y_rf = rf_data["fitness"].values

# Global RF
rf = RandomForestRegressor(n_estimators=200, random_state=42, oob_score=True, n_jobs=-1)
rf.fit(X_rf, y_rf)
print(f"Global RF OOB R^2: {rf.oob_score_:.4f}")

# Permutation importance
perm_imp = permutation_importance(rf, X_rf, y_rf, n_repeats=10, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": bm_cols,
    "importance": perm_imp.importances_mean,
    "std": perm_imp.importances_std,
}).sort_values("importance", ascending=False)

# Top-20 bar chart
top20 = perm_df.head(20)
fig, ax = plt.subplots(figsize=(12, 8))
colors_rf = [CATEGORY_COLORS.get(feat_to_cat.get(f, "Unknown"), "#333333")
             for f in top20["feature"]]
ax.barh(range(len(top20)), top20["importance"].values,
        xerr=top20["std"].values, color=colors_rf, capsize=3)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f.replace("bm_", "") for f in top20["feature"]], fontsize=9)
ax.set_xlabel("Permutation Importance")
ax.set_title("Top-20 Features by Permutation Importance (Global RF)")
ax.invert_yaxis()
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=cat) for cat, c in CATEGORY_COLORS.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

# --- Per-model RF importance heatmap ---
per_model_imp = pd.DataFrame(index=bm_cols, columns=MODELS, dtype=float)
per_model_r2 = {}
for model in MODELS:
    sub = df_valid[df_valid["model"] == model][bm_cols + ["fitness"]].dropna()
    if len(sub) < 10:
        continue
    X_m = sub[bm_cols].values
    y_m = sub["fitness"].values
    rf_m = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
    rf_m.fit(X_m, y_m)
    per_model_r2[model] = rf_m.oob_score_
    pi = permutation_importance(rf_m, X_m, y_m, n_repeats=10, random_state=42, n_jobs=-1)
    per_model_imp[model] = pi.importances_mean

# Sort by global importance
sort_idx = perm_df["feature"].tolist()
per_model_imp_sorted = per_model_imp.loc[sort_idx].head(20)

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(per_model_imp_sorted.astype(float).values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(MODELS)))
ax.set_yticks(range(len(per_model_imp_sorted)))
ax.set_xticklabels(MODELS, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels([f.replace("bm_", "") for f in per_model_imp_sorted.index], fontsize=9)
plt.colorbar(im, ax=ax, shrink=0.8, label="Permutation Importance")
ax.set_title("Per-Model Permutation Importance (Top-20 by Global)")
plt.tight_layout()
plt.show()

print("\nPer-model OOB R^2:")
for m, r2 in per_model_r2.items():
    print(f"  {m}: {r2:.4f}")

In [ ]:
# === Cell 5: SHAP Analysis ===

if not HAS_SHAP:
    print("SHAP not available. Install with: pip install shap")
else:
    # Global SHAP
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_rf)

    feature_names_clean = [c.replace("bm_", "") for c in bm_cols]

    # Beeswarm summary plot
    fig, ax = plt.subplots(figsize=(12, 10))
    shap.summary_plot(shap_values, X_rf, feature_names=feature_names_clean,
                      show=False, max_display=20)
    plt.title("SHAP Summary (Beeswarm) - Global RF")
    plt.tight_layout()
    plt.show()

    # Top-5 SHAP dependence plots
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top5_idx = np.argsort(mean_abs_shap)[::-1][:5]

    for idx in top5_idx:
        fig, ax = plt.subplots(figsize=(8, 5))
        shap.dependence_plot(idx, shap_values, X_rf,
                             feature_names=feature_names_clean,
                             show=False, ax=ax)
        plt.tight_layout()
        plt.show()

    # Per-model SHAP: mean |SHAP| per feature per model
    shap_per_model = pd.DataFrame(index=bm_cols, columns=MODELS, dtype=float)
    for model in MODELS:
        sub = df_valid[df_valid["model"] == model][bm_cols + ["fitness"]].dropna()
        if len(sub) < 10:
            continue
        X_m = sub[bm_cols].values
        y_m = sub["fitness"].values
        rf_m = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_m.fit(X_m, y_m)
        exp_m = shap.TreeExplainer(rf_m)
        sv_m = exp_m.shap_values(X_m)
        shap_per_model[model] = np.abs(sv_m).mean(axis=0)

    # Heatmap
    shap_top = shap_per_model.loc[perm_df["feature"].head(20).tolist()]
    fig, ax = plt.subplots(figsize=(14, 10))
    im = ax.imshow(shap_top.astype(float).values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(MODELS)))
    ax.set_yticks(range(len(shap_top)))
    ax.set_xticklabels(MODELS, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels([f.replace("bm_", "") for f in shap_top.index], fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8, label="Mean |SHAP value|")
    ax.set_title("Per-Model Mean |SHAP| (Top-20 Features)")
    plt.tight_layout()
    plt.show()

In [ ]:
# === Cell 6: Partial Dependence Plots ===

# Use top-5 features from permutation importance
top5_features = perm_df["feature"].head(5).tolist()
top5_indices = [bm_cols.index(f) for f in top5_features]

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
display = PartialDependenceDisplay.from_estimator(
    rf, X_rf, features=top5_indices,
    feature_names=[c.replace("bm_", "") for c in bm_cols],
    ax=axes, kind="average",
)
fig.suptitle("Partial Dependence Plots - Top 5 Features", fontsize=14)
plt.tight_layout()
plt.show()

# 2D interaction PDP for top-2 features
if len(top5_indices) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    display_2d = PartialDependenceDisplay.from_estimator(
        rf, X_rf, features=[(top5_indices[0], top5_indices[1])],
        feature_names=[c.replace("bm_", "") for c in bm_cols],
        ax=ax, kind="average",
    )
    ax.set_title("2D Partial Dependence: Top-2 Interacting Features")
    plt.tight_layout()
    plt.show()

In [ ]:
# === Cell 7: Top vs Bottom Distributional Analysis ===

q75 = df_valid["fitness"].quantile(0.75)
q25 = df_valid["fitness"].quantile(0.25)
top_25 = df_valid[df_valid["fitness"] >= q75]
bot_25 = df_valid[df_valid["fitness"] <= q25]

print(f"Top 25% threshold: {q75:.4f} (n={len(top_25)})")
print(f"Bottom 25% threshold: {q25:.4f} (n={len(bot_25)})")

# KS tests
ks_results = []
for col in bm_cols:
    t = top_25[col].dropna()
    b = bot_25[col].dropna()
    if len(t) > 3 and len(b) > 3:
        ks_stat, ks_p = stats.ks_2samp(t, b)
        ks_results.append({"feature": col, "ks_stat": ks_stat, "p": ks_p,
                           "category": feat_to_cat.get(col, "Unknown")})

ks_df = pd.DataFrame(ks_results).sort_values("ks_stat", ascending=False)

print("\nKS Test Results (sorted by effect size):")
print(f"{'Feature':45s} {'KS stat':>10s} {'p-value':>12s}")
print("-" * 70)
for _, row in ks_df.iterrows():
    marker = " <<<" if row["ks_stat"] > 0.3 else ""
    print(f"{row['feature'].replace('bm_', ''):45s} {row['ks_stat']:10.4f} {row['p']:12.2e}{marker}")

# Violin plots for top features (KS > 0.3 or top-10)
highlight = ks_df[ks_df["ks_stat"] > 0.3]["feature"].tolist()
if len(highlight) < 4:
    highlight = ks_df.head(10)["feature"].tolist()
else:
    highlight = highlight[:12]  # cap at 12

n_plots = len(highlight)
ncols = min(4, n_plots)
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_1d(axes).flatten()

for i, feat in enumerate(highlight):
    ax = axes[i]
    data_vp = [top_25[feat].dropna().values, bot_25[feat].dropna().values]
    parts = ax.violinplot(data_vp, showmedians=True)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["Top 25%", "Bottom 25%"])
    ks_row = ks_df[ks_df["feature"] == feat].iloc[0]
    ax.set_title(f"{feat.replace('bm_', '')}\nKS={ks_row['ks_stat']:.3f}, p={ks_row['p']:.2e}",
                 fontsize=10)

for i in range(n_plots, len(axes)):
    axes[i].set_visible(False)

fig.suptitle("Top 25% vs Bottom 25% AOCC: Feature Distributions", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 8: Behavioral Space Projections ===

# Prepare data: standardize, drop NaN rows
proj_data = df_valid[bm_cols + ["fitness", "model"]].dropna(subset=bm_cols)
X_proj = proj_data[bm_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_proj)

# --- PCA biplot ---
pca = PCA(n_components=min(10, X_scaled.shape[1]))
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 9))
model_list = proj_data["model"].values
unique_models = sorted(set(model_list))
cmap_models = plt.cm.get_cmap("tab10", len(unique_models))

# Normalize fitness for marker size
fitness_vals = proj_data["fitness"].values
fit_norm = (fitness_vals - fitness_vals.min()) / (fitness_vals.max() - fitness_vals.min() + 1e-12)
sizes = 20 + 150 * fit_norm

for i, model in enumerate(unique_models):
    mask = model_list == model
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=sizes[mask],
               c=[cmap_models(i)], label=model, alpha=0.6, edgecolors="w", linewidths=0.3)
    # Convex hull
    pts = X_pca[mask, :2]
    if len(pts) >= 3:
        try:
            hull = ConvexHull(pts)
            hull_pts = np.append(hull.vertices, hull.vertices[0])
            ax.plot(pts[hull_pts, 0], pts[hull_pts, 1], '-', color=cmap_models(i), alpha=0.4, lw=1)
        except Exception:
            pass

# Loading arrows
loadings = pca.components_[:2].T
scale_factor = np.abs(X_pca[:, :2]).max() * 0.8
# Show top-5 loadings
loading_mag = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
top_load_idx = np.argsort(loading_mag)[::-1][:5]
for idx in top_load_idx:
    ax.annotate("", xy=(loadings[idx, 0] * scale_factor, loadings[idx, 1] * scale_factor),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
    ax.text(loadings[idx, 0] * scale_factor * 1.1, loadings[idx, 1] * scale_factor * 1.1,
            bm_cols[idx].replace("bm_", ""), fontsize=8, color="red")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title("PCA Biplot of Behavioral Space (size ~ AOCC)")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

# Variance explained
print("Variance explained per PC:")
for i, ve in enumerate(pca.explained_variance_ratio_[:10]):
    cum = pca.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1}: {ve:.3f} (cumulative: {cum:.3f})")

# Top-5 loadings per PC
for pc in range(min(3, pca.n_components_)):
    abs_loadings = np.abs(pca.components_[pc])
    top5_load = np.argsort(abs_loadings)[::-1][:5]
    print(f"\nPC{pc+1} top loadings:")
    for idx in top5_load:
        print(f"  {bm_cols[idx].replace('bm_', ''):40s} {pca.components_[pc, idx]:+.3f}")

# --- t-SNE ---
n_samples = X_scaled.shape[0]
perp = min(30, n_samples - 1)
tsne = TSNE(n_components=2, perplexity=perp, random_state=42, init="pca")
X_tsne = tsne.fit_transform(X_scaled)

# Model family markers
model_families = {}
markers_map = {"qwen": "o", "rnj": "s", "devstral": "D", "olmo": "^", "granite": "v", "gemini": "*"}
for m in unique_models:
    for prefix, marker in markers_map.items():
        if m.startswith(prefix):
            model_families[m] = marker
            break
    else:
        model_families[m] = "o"

fig, ax = plt.subplots(figsize=(12, 9))
for model in unique_models:
    mask = model_list == model
    sc = ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=fitness_vals[mask], cmap="viridis",
                    marker=model_families.get(model, "o"),
                    s=50, alpha=0.7, label=model,
                    vmin=fitness_vals.min(), vmax=fitness_vals.max(),
                    edgecolors="w", linewidths=0.3)
plt.colorbar(sc, ax=ax, label="AOCC (fitness)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE of Behavioral Space (color = AOCC)")
ax.legend(fontsize=7, loc="best", ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 9: Behavioral Profile Clustering ===

# Standardize
clust_data = df_valid[bm_cols + ["fitness", "model"]].dropna(subset=bm_cols).copy()
X_clust = StandardScaler().fit_transform(clust_data[bm_cols].values)

# Elbow + silhouette
K_range = range(2, 8)
inertias = []
silhouettes = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(list(K_range), inertias, "bo-")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")
ax1.set_title("Elbow Plot")
ax2.plot(list(K_range), silhouettes, "ro-")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("Silhouette Score")
plt.tight_layout()
plt.show()

# Choose best k by silhouette
best_k = list(K_range)[np.argmax(silhouettes)]
print(f"Best k by silhouette: {best_k}")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
clust_data["cluster"] = km_final.fit_predict(X_clust)

# Cluster characterization
print("\nCluster characteristics:")
for c in range(best_k):
    sub = clust_data[clust_data["cluster"] == c]
    print(f"\n--- Cluster {c} (n={len(sub)}) ---")
    print(f"  Mean AOCC: {sub['fitness'].mean():.4f} +/- {sub['fitness'].std():.4f}")
    print(f"  Model composition: {sub['model'].value_counts().to_dict()}")

# Cross-tab: cluster x model
ct = pd.crosstab(clust_data["cluster"], clust_data["model"])
chi2, p_chi, dof, expected = stats.chi2_contingency(ct)
print(f"\nChi-squared test (cluster x model): chi2={chi2:.2f}, p={p_chi:.4e}, dof={dof}")
print("\nCross-tabulation:")
print(ct.to_string())

# Radar/spider plots per cluster
# Find top-10 most discriminative features (by ANOVA F across clusters)
f_scores = []
for i, col in enumerate(bm_cols):
    groups = [clust_data[clust_data["cluster"] == c][col].dropna().values
              for c in range(best_k)]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        f, p = stats.f_oneway(*groups)
        f_scores.append((col, f))
f_scores.sort(key=lambda x: x[1], reverse=True)
top10_disc = [x[0] for x in f_scores[:10]]

# Normalize cluster means for radar
cluster_means = clust_data.groupby("cluster")[top10_disc].mean()
scaler_radar = MinMaxScaler()
cluster_means_norm = pd.DataFrame(
    scaler_radar.fit_transform(cluster_means),
    columns=top10_disc, index=cluster_means.index
)

n_feat = len(top10_disc)
angles = np.linspace(0, 2 * np.pi, n_feat, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors_cluster = plt.cm.Set2(np.linspace(0, 1, best_k))
for c in range(best_k):
    values = cluster_means_norm.loc[c].tolist()
    values += values[:1]
    ax.plot(angles, values, "o-", color=colors_cluster[c],
            label=f"Cluster {c} (n={len(clust_data[clust_data['cluster']==c])})")
    ax.fill(angles, values, alpha=0.15, color=colors_cluster[c])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace("bm_", "") for f in top10_disc], fontsize=8)
ax.set_title("Behavioral Profile by Cluster (Top-10 Discriminative Features)", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 10: Behavioral Evolution Over Generations ===

# Top-10 features by global importance
top10_feats = perm_df["feature"].head(10).tolist()

# Track best-so-far candidate's behavioral features over evaluations
def get_best_so_far_features(model, seed, features):
    """For each evaluation, track the behavioral features of the best-so-far candidate."""
    sub = df[(df["model"] == model) & (df["seed"] == seed)].sort_values("evaluation")
    best_fitness = float("-inf")
    best_features = {f: np.nan for f in features}
    trajectory = []
    for _, row in sub.iterrows():
        f = parse_fitness(row.get("fitness"))
        if f > best_fitness:
            best_fitness = f
            for feat in features:
                if pd.notna(row.get(feat)):
                    best_features[feat] = row[feat]
        traj_row = {"evaluation": row["evaluation"], "best_fitness": best_fitness}
        traj_row.update(best_features)
        trajectory.append(traj_row)
    return pd.DataFrame(trajectory)

# Compute trajectories for all models
trajectories = {}
for model in MODELS:
    model_trajs = []
    for seed in range(N_SEEDS):
        traj = get_best_so_far_features(model, seed, top10_feats)
        if len(traj) > 0:
            model_trajs.append(traj)
    if model_trajs:
        trajectories[model] = model_trajs

# Per model: line plot of each metric's best-so-far value (mean +/- std across seeds)
fig, axes = plt.subplots(2, 5, figsize=(25, 10), sharex=True)
axes = axes.flatten()

for feat_idx, feat in enumerate(top10_feats):
    ax = axes[feat_idx]
    for model in MODELS:
        if model not in trajectories:
            continue
        # Align by evaluation index, compute mean/std
        all_vals = []
        for traj in trajectories[model]:
            if feat in traj.columns:
                all_vals.append(traj.set_index("evaluation")[feat])
        if not all_vals:
            continue
        combined = pd.concat(all_vals, axis=1)
        mean_vals = combined.mean(axis=1)
        std_vals = combined.std(axis=1)
        ax.plot(mean_vals.index, mean_vals.values, label=model, alpha=0.8, lw=1.2)
        ax.fill_between(mean_vals.index,
                        (mean_vals - std_vals).values,
                        (mean_vals + std_vals).values, alpha=0.1)
    ax.set_title(feat.replace("bm_", ""), fontsize=10)
    ax.set_xlabel("Evaluation")
    if feat_idx == 0:
        ax.legend(fontsize=6, loc="best")

fig.suptitle("Best-So-Far Behavioral Features Over Evaluations", fontsize=14)
plt.tight_layout()
plt.show()

# --- Side-by-side: top-3 vs bottom-3 models for most important feature ---
model_perf = df_valid.groupby("model")["fitness"].mean().sort_values(ascending=False)
top3_models = model_perf.head(3).index.tolist()
bot3_models = model_perf.tail(3).index.tolist()

top3_feats = top10_feats[:3]
fig, axes = plt.subplots(len(top3_feats), 2, figsize=(16, 4 * len(top3_feats)),
                         sharex=True)
for row_i, feat in enumerate(top3_feats):
    for col_i, (group, group_label) in enumerate([(top3_models, "Top-3 Models"),
                                                   (bot3_models, "Bottom-3 Models")]):
        ax = axes[row_i, col_i]
        for model in group:
            if model not in trajectories:
                continue
            all_vals = []
            for traj in trajectories[model]:
                if feat in traj.columns:
                    all_vals.append(traj.set_index("evaluation")[feat])
            if not all_vals:
                continue
            combined = pd.concat(all_vals, axis=1)
            mean_vals = combined.mean(axis=1)
            std_vals = combined.std(axis=1)
            ax.plot(mean_vals.index, mean_vals.values, label=model, lw=1.5)
            ax.fill_between(mean_vals.index,
                            (mean_vals - std_vals).values,
                            (mean_vals + std_vals).values, alpha=0.15)
        ax.set_title(f"{feat.replace('bm_', '')} - {group_label}", fontsize=10)
        ax.legend(fontsize=8)
        ax.set_xlabel("Evaluation")

fig.suptitle("Top-3 vs Bottom-3 Models: Key Feature Evolution", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# === Cell 11: Summary of Behavioral Insights ===

# --- Consensus ranking: Borda count ---
# Rank 1: Spearman |rho| with AOCC
rank_spearman = rho_df.set_index("feature")["rho"].abs().rank(ascending=False)

# Rank 2: RF permutation importance
rank_rf = perm_df.set_index("feature")["importance"].rank(ascending=False)

# Rank 3: SHAP (if available)
if HAS_SHAP:
    shap_importance = pd.Series(np.abs(shap_values).mean(axis=0), index=bm_cols)
    rank_shap = shap_importance.rank(ascending=False)
else:
    rank_shap = rank_rf.copy()  # fallback: duplicate RF rank
    print("SHAP not available; using RF importance for SHAP rank.")

# Borda count (lower = better)
common_feats = sorted(set(rank_spearman.index) & set(rank_rf.index) & set(rank_shap.index))
consensus = pd.DataFrame({
    "spearman_rank": [rank_spearman.get(f, len(bm_cols)) for f in common_feats],
    "rf_rank": [rank_rf.get(f, len(bm_cols)) for f in common_feats],
    "shap_rank": [rank_shap.get(f, len(bm_cols)) for f in common_feats],
}, index=common_feats)
consensus["borda"] = consensus.sum(axis=1)
consensus = consensus.sort_values("borda")
consensus["category"] = [feat_to_cat.get(f, "Unknown") for f in consensus.index]

print("=" * 90)
print("CONSENSUS RANKING (Top-10 Behavioral Features)")
print("=" * 90)
print(f"{'Feature':42s} {'Spearman':>9s} {'RF':>6s} {'SHAP':>6s} {'Borda':>7s} {'Category'}")
print("-" * 90)
for feat in consensus.head(10).index:
    row = consensus.loc[feat]
    name = feat.replace("bm_", "")
    print(f"{name:42s} {row['spearman_rank']:9.0f} {row['rf_rank']:6.0f} "
          f"{row['shap_rank']:6.0f} {row['borda']:7.0f} {row['category']}")

# --- Key findings per feature category ---
print("\n" + "=" * 90)
print("KEY FINDINGS PER CATEGORY")
print("=" * 90)
for cat in FEATURE_CATEGORIES:
    cat_feats = [f"bm_{f}" for f in FEATURE_CATEGORIES[cat] if f"bm_{f}" in consensus.index]
    if not cat_feats:
        continue
    cat_consensus = consensus.loc[cat_feats].sort_values("borda")
    best = cat_consensus.index[0].replace("bm_", "")
    best_borda = cat_consensus.iloc[0]["borda"]
    avg_borda = cat_consensus["borda"].mean()
    # Get global rho for best
    rho_val = rho_df[rho_df["feature"] == cat_consensus.index[0]]["rho"].values
    rho_str = f"{rho_val[0]:+.3f}" if len(rho_val) > 0 else "N/A"
    print(f"\n{cat}:")
    print(f"  Best feature: {best} (Borda={best_borda:.0f}, rho={rho_str})")
    print(f"  Category avg Borda: {avg_borda:.1f}")
    print(f"  Features: {', '.join(f.replace('bm_', '') for f in cat_feats)}")

# --- Per-model behavioral fingerprint ---
print("\n" + "=" * 90)
print("PER-MODEL BEHAVIORAL FINGERPRINT (top-5 consensus features)")
print("=" * 90)
top5_consensus = consensus.head(5).index.tolist()
for model in MODELS:
    sub = df_valid[df_valid["model"] == model]
    if len(sub) == 0:
        continue
    mean_aocc = sub["fitness"].mean()
    print(f"\n{model} (mean AOCC={mean_aocc:.4f}, n={len(sub)}):")
    for feat in top5_consensus:
        vals = sub[feat].dropna()
        if len(vals) > 0:
            name = feat.replace("bm_", "")
            print(f"  {name:40s} mean={vals.mean():.4f}  std={vals.std():.4f}")